# Snowpark 4DGS — Colab Free pilot

A one-second synthetic research experiment: 16 cameras, 31 times, 496 calibrated images.
Compare a static control and a time-conditioned model on separate spatial/temporal test groups.

Select **Runtime → Change runtime type → T4 GPU**, standard RAM. Use **Colab Free** only;
no cell purchases credits, upgrades a plan, opens SSH, or keeps an idle session alive.
This backend contains Inria research/evaluation components and is not commercially cleared.

First prepare the input bundle locally as described in `docs/4dgs.md`. Only synthetic RGB,
evaluation masks, calibration and the pilot's code enter the bundle; personal reference images
and the rest of Drive are not needed. Runtime storage is ephemeral: download the results.

In [ ]:
import datetime
import hashlib
import json
import os
import pathlib
import platform
import subprocess
import sys
import time

import torch

assert torch.cuda.is_available(), "Select a free T4 GPU runtime"
assert "T4" in torch.cuda.get_device_name(0), (
    "This environment was tested with T4; review another GPU explicitly"
)
SESSION_STARTED = time.monotonic()
RUN = pathlib.Path("/content") / (
    "snowpark-4dgs-" + datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S-%fZ")
)
RUN.mkdir(exist_ok=False)


def checked(command, log_name, timeout=1800, env=None, cwd=None):
    remaining = 10800 - (time.monotonic() - SESSION_STARTED)
    if remaining <= 0:
        raise RuntimeError("Three-hour session budget exhausted")
    print("START", log_name, flush=True)
    started = time.monotonic()
    with (RUN / log_name).open("x") as log:
        process = subprocess.run(
            list(map(str, command)),
            cwd=cwd,
            env=env,
            stdout=log,
            stderr=subprocess.STDOUT,
            timeout=min(timeout, remaining),
        )
    print(
        "FINISH",
        log_name,
        "exit",
        process.returncode,
        "seconds",
        round(time.monotonic() - started, 2),
        flush=True,
    )
    if process.returncode:
        print((RUN / log_name).read_text()[-8000:])
        raise RuntimeError(log_name + " failed; preserve the log and stop")


report = dict(
    python=sys.version,
    platform=platform.platform(),
    torch=str(torch.__version__),
    torch_cuda=torch.version.cuda,
    gpu=torch.cuda.get_device_name(0),
)
for name, command in {
    "gpu": [
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.free,driver_version",
        "--format=csv",
    ],
    "compiler": ["nvcc", "--version"],
}.items():
    process = subprocess.run(command, capture_output=True, text=True)
    report[name] = dict(exit_code=process.returncode, stdout=process.stdout, stderr=process.stderr)
    assert process.returncode == 0
(RUN / "gpu-preflight.json").write_text(json.dumps(report, indent=2))
print(json.dumps(report, indent=2))

## Isolated Python/CUDA environment

The current notebook kernel may be much newer than the research code. Keep it intact;
run the backend in its own Python 3.10 environment with Torch 1.13.1/CUDA 11.7 wheels
and a compatible CUDA 11.8 compiler. Install legacy build tools **before** MMCV.
The original `simple-knn` component is not built or executed: the adapter uses native Torch.

In [ ]:
import tarfile
import urllib.request

ENV = RUN / "env"
MAMBA_ENV = dict(os.environ, MAMBA_ROOT_PREFIX=str(RUN / "mamba"), MAMBA_NO_BANNER="1")
archive = RUN / "micromamba.tar.bz2"
urllib.request.urlretrieve("https://micro.mamba.pm/api/micromamba/linux-64/latest", archive)
with tarfile.open(archive, "r:bz2") as stream:
    (RUN / "micromamba").write_bytes(stream.extractfile("bin/micromamba").read())
(RUN / "micromamba").chmod(0o700)
checked(
    [
        RUN / "micromamba",
        "create",
        "--yes",
        "--no-rc",
        "--prefix",
        ENV,
        "-c",
        "nvidia/label/cuda-11.8.0",
        "-c",
        "conda-forge",
        "python=3.10",
        "pip<25",
        "cuda-nvcc=11.8",
        "cuda-cudart=11.8",
        "cuda-cudart-dev=11.8",
        "cuda-cccl=11.8",
        "gxx_linux-64=11",
        "sysroot_linux-64=2.17",
    ],
    "environment-create.log",
    timeout=5400,
    env=MAMBA_ENV,
)
PYTHON = ENV / "bin/python"
checked(
    [PYTHON, "-m", "pip", "install", "setuptools==69.5.1", "numpy==1.24.4", "wheel"],
    "build-tools.log",
    timeout=300,
)
checked(
    [
        PYTHON,
        "-m",
        "pip",
        "install",
        "torch==1.13.1+cu117",
        "torchvision==0.14.1+cu117",
        "numpy==1.24.4",
        "--extra-index-url",
        "https://download.pytorch.org/whl/cu117",
    ],
    "torch-install.log",
)
checked(
    [
        PYTHON,
        "-m",
        "pip",
        "install",
        "ninja==1.11.1.1",
        "mmcv==1.6.0",
        "matplotlib==3.7.5",
        "Pillow==10.4.0",
        "imageio==2.35.1",
        "imageio-ffmpeg==0.5.1",
        "open3d==0.18.0",
        "tqdm==4.66.5",
        "plyfile==1.0.3",
        "pytorch-msssim==1.0.0",
        "lpips==0.1.4",
        "scikit-image==0.21.0",
        "opencv-python-headless==4.10.0.84",
        "scipy==1.10.1",
        "yapf==0.40.1",
    ],
    "dependencies-install.log",
)
BACKEND_ENV = dict(
    os.environ,
    CUDA_HOME=str(ENV),
    PATH=str(ENV / "bin") + ":" + os.environ["PATH"],
    CXX=str(ENV / "bin/x86_64-conda-linux-gnu-c++"),
    CC=str(ENV / "bin/x86_64-conda-linux-gnu-cc"),
    TORCH_CUDA_ARCH_LIST="7.5",
    MAX_JOBS="2",
)
checked(
    [
        PYTHON,
        "-c",
        "import torch,numpy,mmcv,open3d; print(torch.__version__,torch.version.cuda,numpy.__version__); assert torch.cuda.is_available()",
    ],
    "isolated-environment-probe.log",
    timeout=120,
    env=BACKEND_ENV,
)
checked([PYTHON, "-m", "pip", "freeze"], "pip-freeze.txt", timeout=120)
checked(
    [RUN / "micromamba", "list", "--prefix", ENV, "--explicit"],
    "conda-explicit.txt",
    timeout=120,
    env=MAMBA_ENV,
)

## Pinned backend and rasterizer

Only the necessary rasterizer/GLM submodules are initialized. The parent repository's
Apache license does not override the rasterizer's Inria research/evaluation restriction.
The adapter preserves copyright headers and records its exact patch.

In [ ]:
BACKEND = RUN / "4DGaussians"
PIN = "843d5ac636c37e4b611242287754f3d4ed150144"
checked(
    [
        "git",
        "clone",
        "--filter=blob:none",
        "--no-checkout",
        "https://github.com/hustvl/4DGaussians.git",
        BACKEND,
    ],
    "backend-clone.log",
    timeout=300,
)
checked(["git", "-C", BACKEND, "checkout", PIN], "backend-checkout.log", timeout=300)
checked(
    [
        "git",
        "-C",
        BACKEND,
        "submodule",
        "update",
        "--init",
        "--recursive",
        "submodules/depth-diff-gaussian-rasterization",
    ],
    "backend-submodules.log",
    timeout=600,
)
checked(
    [
        PYTHON,
        "-m",
        "pip",
        "install",
        "--no-build-isolation",
        BACKEND / "submodules/depth-diff-gaussian-rasterization",
    ],
    "rasterizer-build.log",
    env=BACKEND_ENV,
)
checked(
    [PYTHON, "-c", 'import torch,diff_gaussian_rasterization; print("RASTERIZER_IMPORT_PASS")'],
    "rasterizer-import.log",
    timeout=120,
    env=BACKEND_ENV,
)
licenses = {}
for folder in [BACKEND, BACKEND / "submodules/depth-diff-gaussian-rasterization"]:
    licenses[str(folder.relative_to(RUN))] = {
        p.name: p.read_text()
        for p in folder.iterdir()
        if p.is_file() and p.name.lower().startswith("license")
    }
(RUN / "backend-licenses.json").write_text(json.dumps(licenses, indent=2))

## Upload the verified input bundle

Upload **pilot-input.zip** generated locally. This does not mount or search Drive.
Extraction rejects path traversal, links, duplicate entries, oversized payloads and
unexpected roots. Final-test records are excluded from the fitting JSON files.

In [ ]:
import stat
import zipfile

from google.colab import files

uploaded = files.upload()
assert len(uploaded) == 1, "Select exactly one pilot-input.zip"
name, payload = next(iter(uploaded.items()))
assert name.endswith(".zip")
bundle_hash = hashlib.sha256(payload).hexdigest()
WORK = RUN / "bundle"
WORK.mkdir(exist_ok=False)
with zipfile.ZipFile(pathlib.Path(name)) as archive:
    names = [item.filename for item in archive.infolist()]
    assert len(names) == len(set(names)), "Duplicate archive entry"
    assert sum(item.file_size for item in archive.infolist()) <= 2 * 1024**3
    for item in archive.infolist():
        target = (WORK / item.filename).resolve()
        assert target.is_relative_to(WORK.resolve())
        assert pathlib.PurePosixPath(item.filename).parts[0] in ("code", "data")
        assert not stat.S_ISLNK(item.external_attr >> 16)
    archive.extractall(WORK)
DATA = WORK / "data"
CODE = WORK / "code"
for relative, expected in json.loads((DATA / "file-hashes.json").read_text()).items():
    path = (DATA / relative).resolve()
    assert path.is_relative_to(DATA.resolve())
    assert hashlib.sha256(path.read_bytes()).hexdigest() == expected, relative
(RUN / "input-bundle.json").write_text(
    json.dumps(dict(filename=name, sha256=bundle_hash, bytes=len(payload)), indent=2)
)
assert json.loads((DATA / "preparation.json").read_text())["counts"] == dict(
    train=300, validation=62, test_spatial=62, test_temporal=72
)
checked(
    [
        PYTHON,
        CODE / "tools/colab_4dgs.py",
        "adapt",
        "--backend",
        BACKEND,
        "--output",
        RUN / "adapter",
    ],
    "adapter.log",
    timeout=120,
    env=BACKEND_ENV,
)

## Real model/memory gate, then the two bounded fits

The diagnostic runs 100 CUDA render/backpropagation steps, loading the actual
300 training and 62 validation images; it never loads final-test images. Diagnostic
weights are discarded. Each production model gets 500 coarse + 4,500 fine steps,
a 30-minute timeout, 12 GiB allocated-memory stop and 200,000-Gaussian stop.
A limit or failed check stops the notebook; it does not silently shrink the experiment.

In [ ]:
RUNNER = CODE / "tools/colab_4dgs.py"
checked(
    [
        PYTHON,
        RUNNER,
        "probe",
        "--backend",
        BACKEND,
        "--data",
        DATA,
        "--output",
        RUN / "memory-smoke",
    ],
    "model-memory-smoke.log",
    timeout=600,
    env=BACKEND_ENV,
)
smoke = json.loads((RUN / "memory-smoke/memory-smoke.json").read_text())
assert smoke["status"] == "PASS_CUDA_RENDER_BACKPROP_100_STEPS"
assert smoke["peak_allocated_bytes"] <= 12 * 1024**3
print(json.dumps(smoke, indent=2))
for mode in ["static", "dynamic"]:
    checked(
        [
            PYTHON,
            RUNNER,
            "fit",
            "--backend",
            BACKEND,
            "--data",
            DATA,
            "--mode",
            mode,
            "--output",
            RUN / mode,
        ],
        mode + "-fit.log",
        timeout=1860,
        env=BACKEND_ENV,
    )

## Sealed evaluation and comparison

Only now open final spatial/temporal views. Render from the same saved model at
multiple times, save predictions/comparison videos, and apply the preregistered
athlete-ROI thresholds. A quantitative pass still requires visual inspection.

In [ ]:
EVALUATOR = CODE / "tools/evaluate_4dgs.py"
for mode in ["static", "dynamic"]:
    checked(
        [
            PYTHON,
            EVALUATOR,
            "evaluate",
            "--backend",
            BACKEND,
            "--data",
            DATA,
            "--model",
            RUN / mode,
            "--mode",
            mode,
            "--output",
            RUN / ("evaluation-" + mode),
        ],
        mode + "-evaluation.log",
        timeout=600,
        env=BACKEND_ENV,
    )
checked(
    [
        PYTHON,
        EVALUATOR,
        "compare",
        "--data",
        DATA,
        "--static",
        RUN / "evaluation-static",
        "--dynamic",
        RUN / "evaluation-dynamic",
        "--output",
        RUN / "comparison.json",
    ],
    "comparison.log",
    timeout=120,
    env=BACKEND_ENV,
)
print((RUN / "comparison.json").read_text())

## Download results before leaving the runtime

The archive includes logs, model checkpoints, the exact adapter patch, metrics and
comparison videos. It excludes the environment, caches, upstream Git checkout and
input images already held locally. No credentials or personal references are packaged.
Keep this downloaded archive in a **new** local output directory.

In [ ]:
archive_path = RUN / (RUN.name + "-results.zip")
folders = [
    "adapter",
    "memory-smoke",
    "static",
    "dynamic",
    "evaluation-static",
    "evaluation-dynamic",
]
with zipfile.ZipFile(archive_path, "x", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN.iterdir()):
        if path.is_file() and path.suffix in (".json", ".log", ".txt"):
            archive.write(path, path.name)
    for folder in folders:
        for path in sorted((RUN / folder).rglob("*")):
            if path.is_file():
                archive.write(path, str(path.relative_to(RUN)))
print(
    "RESULT_ARCHIVE",
    archive_path.stat().st_size,
    hashlib.sha256(archive_path.read_bytes()).hexdigest(),
)
files.download(str(archive_path))